# Newton's cradle resolved by pairwise impulses

**This notebook is one half of a pair.** The other half is the example
file `physical_object/examples/newtons_cradle.rs` in this repository, which contains the same simulation
written directly in the simulator's own command language. This notebook
runs that same simulation from Python, and explains every line of it.

**This notebook is complete in itself.** Everything needed to understand
and run it is written below. You will never be asked to open another
file to find an explanation.

**The example this notebook pairs with is a compiled Rust program**, `physical_object/examples/newtons_cradle.rs`. It is self-checking: it runs its physics, compares the results against closed-form expectations, prints SUCCESS or FAILURE, and exits nonzero on failure.

This notebook reproduces the same physical situation through the simulator's command language, explains every line of it, and checks the same things: one incomer strikes four touching spheres and only the far sphere leaves, at the incomer's speed — forced by momentum and energy conservation applied pairwise along the line of centres.

The two are the same physics on the same engine: the compiled example calls the `physical_object` library directly, and the commands below drive that same library through the `posim` interpreter. At the end, this notebook also runs the compiled example itself and shows its verdict.

To run the paired example directly, without Python and without Jupyter:

```bash
cargo run --release -p physical_object --example newtons_cradle
```

---


## 1. How to open a notebook like this one, from a terminal

This section is repeated in full in every notebook in this project, so
that you never have to go looking in another file for it.

### 1.1 What you need

| you need | why | check it with |
|---|---|---|
| **Rust** (1.75 or newer) | the simulator is pure Rust and is built from source | `rustc --version` |
| **Python 3.8+** | Jupyter runs on Python, and this notebook is a Python notebook | `python3 --version` |
| **JupyterLab** or **Jupyter Notebook** | the program that opens `.ipynb` files | `jupyter --version` |
| the **posim** binary | the simulator itself; you build it once, below | `ls target/release/posim` |

You do **not** need SUNDIALS installed. You do **not** need a C or Fortran
compiler. You do **not** need any package from crates.io. The whole solver
suite — CVODE, CVODES, IDA, IDAS, KINSOL, ARKODE — is a pure-Rust
translation vendored inside this repository, and `cargo` builds it from
the source that is already on your disk. Nothing is downloaded at build
time.

### 1.2 Build the simulator, once

Open a terminal and run these four commands. `$` is the shell prompt —
do not type it.

```bash
$ git clone https://github.com/once-ere/rustSolveIt_macos-silicon_SUNDIALS_7_8_0.git
$ cd rustSolveIt_macos-silicon_SUNDIALS_7_8_0
$ cargo build --release -p posim
$ ls -l target/release/posim
```

The third command is the long one: it compiles the simulator and the whole
vendored SUNDIALS translation, and takes a few minutes the first time.
When it finishes, the fourth command must print a line describing an
executable file. If it prints `No such file or directory`, the build did
not succeed — scroll up in the terminal and read the first error, because
later errors are usually consequences of it.

### 1.3 Install Jupyter, if you do not have it

```bash
$ python3 -m pip install --user jupyterlab
$ jupyter --version
```

If `jupyter` is still "command not found" after this, your `pip --user`
scripts directory is not on your `PATH`. Print it and add it:

```bash
$ python3 -m site --user-base
$ export PATH="$(python3 -m site --user-base)/bin:$PATH"
```

To make that permanent, append that same `export` line to `~/.bashrc`
(bash) or `~/.zshrc` (zsh).

### 1.4 Start Jupyter and open a NEW notebook

From the repository root:

```bash
$ jupyter lab
```

That starts a small web server and prints a URL that contains a one-time
token, looking like:

```
    http://localhost:8888/lab?token=8f4c1e...
```

It normally opens your browser by itself. If it does not, copy that whole
URL — token included — into a browser. The token is the password; a URL
without it will be refused.

Then, inside JupyterLab:

1. **File → New → Notebook**.
2. When it asks you to *Select Kernel*, choose **Python 3 (ipykernel)**.
3. The empty notebook appears. Anything you type into a cell is Python.

Choose **Python 3**, and not the "posim" kernel that this repository also
ships, because this notebook is written in Python: it starts the simulator
itself, sends it commands, reads the replies back, and (at the end) opens
a graphical save dialog. Those are Python actions. The posim kernel speaks
only the simulator's own command language and cannot do them.

### 1.5 Running cells

- **Shift+Enter** runs the selected cell and moves to the next one.
- **Ctrl+Enter** runs the selected cell and stays put.
- Run the cells of this notebook **in order, from the top**. Each one
  builds on the state left by the ones above it, so skipping one will
  usually produce an error in a later cell.
- If things get into a confusing state, use **Kernel → Restart Kernel and
  Clear All Outputs**, then start again from the first cell.

### 1.6 If you would rather not use Jupyter at all

Everything this notebook does can also be typed straight into the
simulator's own prompt:

```bash
$ cargo run --release -p posim
```

That gives you an `In[1]:=` prompt where the simulator's commands are
typed directly. Type `HELP` for the command reference and `QUIT` to
leave. The Python in this notebook is a wrapper around exactly that
program.

---


## 2. The words used in this notebook

Repeated in full in every notebook of this project, so that this notebook
stands alone.

**Body.** One rigid object. It has a *shape* (which fixes its moments of
inertia), a mass, a position, a velocity, an orientation and an angular
velocity. Bodies are created with the `NEW` command.

**Boundary / shape.** One of `point`, `sphere`, `cuboid`, `cylinder`,
`disk`, `torus`, `dumbbell`. The shape is not decoration: the simulator
computes the body's inertia tensor from it analytically.

**Static body / anchor.** A body with `inverse_mass = 0`. No force can
move it. It is how a mechanism is bolted to the world. A `point` anchor
additionally has a zero inertia tensor, so no torque can turn it either.

**Joint / constraint.** An exact geometric or kinematic relation that the
solver holds true for all time — not a stiff spring. Each joint
contributes a fixed number of scalar equations, called *rows*:

| command | rows | what it holds | freedoms it leaves |
|---|---|---|---|
| `CONSTRAIN a b [len]` | 1 | a fixed distance between two centres | 5 |
| `GEAR a b <axis> <ratio>` | 1 | a fixed proportion between two turns | 5 |
| `RACK p b <axis> <dir> <r>` | 1 | a turn tied to a slide, `ds = r dtheta` | 5 |
| `BALL a b` | 3 | one shared point | 3 (any rotation about it) |
| `UNIVERSAL a b <u> <w>` | 4 | a shared point and a right angle | 2 |
| `HINGE a b <axis>` | 5 | a shared point and a shared axis | 1 (the swing) |
| `PRISMATIC a b <axis>` | 5 | a line to slide along, and no turning | 1 (the slide) |

**Degrees of freedom.** Each free body has 6 (three of position, three of
orientation). A mechanism of `N` free bodies has `6N`. Subtract the total
rows of all its joints and what remains is how many independent ways the
mechanism can still move. If the subtraction leaves zero the mechanism is
locked; if the rows are not independent of one another the system is
*redundant* and the solver will refuse it rather than guess.

**The pivot rule.** Every joint that holds a point places that point at
the **midpoint of the two bodies' centres, as they stand at the moment the
joint is created**, and then remembers it in each body's own frame. So you
position the parts first and join them second, and the geometry you
assemble is the geometry you get.

**`g` and `g_dot`.** `g` is the vector of constraint equations; the solver
holds `g = 0`. `g_dot` is its time derivative, which the solver also holds
at zero. Their sizes are the total row count. The `CONSTRAINTS` command
prints the worst absolute value of each, and those two numbers are how you
tell whether a mechanism is really being held together: they should stay
near the solver's tolerance and never grow steadily.

**`METHOD IDA`.** Any mechanism with a joint must be integrated by IDA,
the differential-algebraic solver, and the simulator refuses any other
method rather than silently integrating an unconstrained problem.

---


## 3. How a second-order mechanics problem becomes a first-order system

Repeated in full in every notebook of this project.

SUNDIALS — like every general ODE/DAE library — integrates **first-order**
systems. Newton's and Euler's equations are **second order**. This section
is the bridge, and it is the same bridge in every example.

### 3.1 The physics, as written by hand

For each body `i`, with mass `m_i`, inertia tensor `I_i` (in the body's own
frame), position `x_i`, orientation `R_i`, and angular velocity `omega_i`:

```
m_i * d2(x_i)/dt2 = F_i                            (Newton)
I_i * d(omega_i)/dt + omega_i x (I_i omega_i) = T_i   (Euler)
```

Both are second order in the configuration variables: the first is second
order in `x_i`, and the second is second order in orientation, because
`omega_i` is itself the rate of change of `R_i`.

### 3.2 The standard trick: name the velocities

A second-order equation becomes two first-order equations by giving the
first derivative its own name and letting the solver carry it as an
unknown alongside the position. This simulator carries **momentum** rather
than velocity, which is the same trick with a better-conditioned variable:

```
d(x_i)/dt = p_i / m_i                    <- was the definition of velocity
d(p_i)/dt = F_i                          <- was Newton's second law
d(q_i)/dt = 0.5 * quat(omega_i) * q_i    <- was the definition of omega
d(L_i)/dt = T_i                          <- was Euler's equation
```

with `omega_i = R_i I_i^-1 R_i^T L_i`, and `q_i` a unit quaternion holding
the orientation. Orientation is stored as a quaternion rather than as
Euler angles because quaternions have no gimbal lock and renormalise
cheaply.

### 3.3 The packed state vector

Each body contributes exactly **13 numbers**, in this order:

```
index 0..2    x       position
index 3..5    p       linear momentum
index 6..9    q       orientation quaternion, w first
index 10..12  L       angular momentum
```

so a model of `N` bodies has a state vector `y` of length `13N`, and the
whole of mechanics above is one first-order system `dy/dt = f(t, y)`.
That is what is handed to the Rust translation of CVODE or ARKODE.

### 3.4 When there are joints: a DAE, not an ODE

A joint is an algebraic relation between coordinates, not a rate, so it
cannot be written as `dy/dt = ...`. The system becomes *differential-
algebraic*, and this simulator uses the **GGL (Gear-Gupta-Leimkuhler)**
index-2 formulation, which carries **both** the position-level constraint
and its derivative, with a Lagrange multiplier for each:

```
d(x)/dt = v - M^-1 J^T mu        position, corrected by the multiplier mu
d(p)/dt = F + J^T lambda         momentum, plus the constraint force
0       = g(q)                   the joints hold, at the position level
0       = J u                    the joints hold, at the velocity level
```

`J` is the constraint Jacobian: row `k` of `J` is the gradient of the
`k`-th constraint equation with respect to all the coordinates, so `J u`
is the rate of change of the constraints under the current velocities.
`lambda` are the constraint forces, `mu` the position-level correction,
and `M^-1` the inverse mass/inertia metric — which is **not** optional,
because `J` has rows in different units and the correction has to be
mass-weighted to be dimensionally consistent.

Carrying both `g` and `J u` is what keeps the joints exact. A formulation
that enforced only the acceleration level would let `g` drift away
quadratically in time, and a mechanism would slowly come apart.

The unknowns are therefore `y` (the `13N` numbers above) together with
`lambda` and `mu` (one of each per constraint row), and the whole thing is
handed to the Rust translation of **IDA**, which solves systems of the
implicit form `F(t, y, y') = 0`.

### 3.5 Consistent initial conditions

A DAE cannot be started from just any state. The starting state must
already satisfy `g = 0` **and** `J u = 0` — the parts must be assembled,
and their initial velocities must be compatible with the joints. If the
velocities are not compatible, this simulator projects them onto the
nearest compatible set before starting, and reports how much it had to
change them. A start that needs no projection is one where you specified
velocities the mechanism can actually have.

---


## 4. The physical situation

### The objects

| name | shape | given properties |
|---|---|---|
| `incomer` | `sphere` | `mass = 1`, `radius = 0.5`, `position = [-3, 0, 0]`, `velocity = [1, 0, 0]` |
| `s1` | `sphere` | `mass = 1`, `radius = 0.5`, `position = [0, 0, 0]` |
| `s2` | `sphere` | `mass = 1`, `radius = 0.5`, `position = [1, 0, 0]` |
| `s3` | `sphere` | `mass = 1`, `radius = 0.5`, `position = [2, 0, 0]` |
| `s4` | `sphere` | `mass = 1`, `radius = 0.5`, `position = [3, 0, 0]` |

Each row is one rigid body. The simulator computes each body's inertia tensor from its shape; you never type an inertia tensor in.

### The shapes, and the inertia each implies

**`sphere`** — a uniform solid ball. `I = diag(2mr^2/5, 2mr^2/5, 2mr^2/5)` — isotropic, so every axis through the centre is a principal axis with the same moment. The only shape whose inertia tensor is unchanged by any rotation, which makes it the simplest collidable body.

### What each property means

- **`mass`** — the body's mass, in kilograms. Setting it also sets `inverse_mass` to its reciprocal — the two are kept consistent, and the simulator carries the inverse because that is what the equations of motion actually use.
- **`radius`** — the radius, in metres.
- **`position`** — the centre of mass, as `[x, y, z]` in metres.
- **`velocity`** — the centre-of-mass velocity, as `[vx, vy, vz]` in metres per second. Setting it sets the linear momentum to `m * v`; the momentum is what the state vector actually carries.

### The interactions

- **Mutual gravity is OFF** (`g_constant = 0`). The simulator's default is 1, so this had to be said explicitly; without it, bodies that are supposed only to bounce or to be held by joints would also attract each other.
- **Contact is ON**. Impacts are located as roots of the separation function by SUNDIALS' rootfinder, so the time of impact is exact to solver tolerance and nothing tunnels through a thin body.

### 4.1 Equations of motion

There are 5 bodies. Each obeys Newton's and Euler's equations, both of which are **second order**:

```
m_i * d2(x_i)/dt2 = F_contact_i

I_i * d(omega_i)/dt + omega_i x (I_i omega_i) = T_i
```

The `omega x (I omega)` term in Euler's equation is the gyroscopic term. It is what makes a tumbling body's spin axis wander even with no torque at all, and it is why rigid-body motion cannot be reduced to three independent rotations.

The forces on the right are:

- `F_contact_i` — impulsive, delivered at the instant of contact along the contact normal, scaled by the coefficient of restitution

### 4.2 Constraint equations

**This model has no joints.** Nothing algebraic relates the coordinates, so the system is an ordinary differential equation rather than a differential-algebraic one, and it needs no Lagrange multipliers and no constraint Jacobian.

That does not mean nothing is conserved. Energy, linear momentum and angular momentum are still exact consequences of the equations of motion, and comparing them before and after a run is the honest way to judge the integration.

### 4.3 The first-order system actually handed to SUNDIALS

**Sizing this particular problem.** There are 5 bodies, so the state vector `y` has

```
5 bodies x 13 numbers = 65 components
```

laid out as consecutive 13-number blocks in creation order, each holding position (3), linear momentum (3), orientation quaternion w-first (4), and angular momentum (3).

**What is handed to the Rust SUNDIALS translation.**

There are no algebraic constraints, so the whole of the mechanics above is one first-order system

```
dy/dt = f(t, y),   y in R^65
```

handed to the pure-Rust translation of **CVODE** in Adams-Moulton mode: a variable-order, variable-step explicit-family method for non-stiff problems. This is the default, and it is the right choice for orbits and for anything smooth.

The right-hand side `f` unpacks `y` into bodies, computes every force and torque, and packs the derivatives back in the same 13-per-body order.

**Contact is handled as rootfinding, not as a force.** The separation between each collidable pair is registered with the solver as a root function. The integrator advances normally until a separation crosses zero, stops exactly there, applies the impulse, and restarts. This is why the time of impact comes out to solver tolerance and why a fast body cannot pass through a thin one between steps.

---


## 5. How this notebook talks to the simulator

Repeated in full in every notebook of this project.

The next cell defines a small helper class. It launches the simulator as a
child process in **machine mode** (`posim --machine`), which makes it speak
JSON Lines: you send it one JSON object per line, and it answers with one
JSON object per line. This is the same interface the project's Jupyter
kernel uses internally.

The three calls used below are:

- `sim.do("...")` — run one simulator command and show what it printed.
  Adding `quiet=True` runs it without printing, which is what you want
  inside a loop that takes thousands of steps.
- `sim.get("obj.field")` — read one value back as a Python object.
- `sim.state()` — fetch the whole simulation state as a Python dictionary.

Nothing here integrates anything: every advance in time is performed by
the Rust SUNDIALS translation inside the child process.


In [1]:
import json, os, shutil, subprocess, sys
from pathlib import Path

def _find_posim():
    """Locate the posim binary, explaining clearly if it is missing."""
    env = os.environ.get("POSIM_BIN")
    if env and Path(env).is_file():
        return env
    onpath = shutil.which("posim")
    if onpath:
        return onpath
    here = Path.cwd()
    for base in [here, *here.parents]:
        for profile in ("release", "debug"):
            cand = base / "target" / profile / "posim"
            if cand.is_file():
                return str(cand)
    raise SystemExit(
        "Could not find the posim binary.\n"
        "Build it first, from the repository root:\n"
        "    cargo build --release -p posim\n"
        "or set the POSIM_BIN environment variable to its full path."
    )

class Sim:
    """One posim child process, spoken to in JSON Lines."""

    def __init__(self):
        self.binary = _find_posim()
        self.proc = subprocess.Popen(
            [self.binary, "--machine"],
            stdin=subprocess.PIPE, stdout=subprocess.PIPE,
            text=True, bufsize=1,
            env=dict(os.environ, POSIM_NO_BROWSER="1"),
        )
        print(f"simulator started: {self.binary}")

    def _rpc(self, obj):
        self.proc.stdin.write(json.dumps(obj) + "\n")
        self.proc.stdin.flush()
        while True:
            line = self.proc.stdout.readline()
            if not line:
                raise RuntimeError("the simulator closed the connection")
            reply = json.loads(line)
            if "event" in reply:      # asynchronous scene notice, not a reply
                continue
            return reply

    def do(self, code, quiet=False):
        """Run one simulator command; print and return what it printed.

        Pass quiet=True inside a loop, where printing a line per step
        would bury the result under thousands of lines of progress.
        """
        r = self._rpc({"op": "exec", "code": code})
        if not r.get("ok"):
            raise RuntimeError(f"the simulator refused {code!r}:\n  {r.get('error')}")
        out = r.get("display") or r.get("result")
        if out not in (None, "") and not quiet:
            print(out)
        return out

    def get(self, path):
        """Read one value, e.g. sim.get('piston.position')."""
        r = self._rpc({"op": "get", "path": path})
        if not r.get("ok"):
            raise RuntimeError(f"could not read {path!r}: {r.get('error')}")
        return r.get("result")

    def state(self):
        """The whole simulation state, as a Python dictionary."""
        r = self._rpc({"op": "state"})
        if not r.get("ok"):
            raise RuntimeError(r.get("error"))
        return r["result"]

    def close(self):
        try:
            self.proc.stdin.write('{"op":"quit"}\n'); self.proc.stdin.flush()
        except (BrokenPipeError, ValueError):
            pass
        self.proc.wait(timeout=30)
        for pipe in (self.proc.stdin, self.proc.stdout):
            try: pipe.close()
            except Exception: pass
        print("simulator stopped")

sim = Sim()

simulator started: /Users/youruser/Developer/github/rustSolveIt_macos-silicon_SUNDIALS_7_8_0/rustSolveIt_macos-silicon_SUNDIALS_7_8_0/target/release/posim


## 6. Building and running the simulation

From here on, the notebook alternates: a section of text explaining
exactly what the next cell asks the simulator to do and what each value
in it means, then the cell itself. Run them in order from here down.


### 6.1 Set up the world

The next cell sends this command to the simulator. Here is exactly what it expects:

---

`set system.g_constant = 0` sets a property of the whole simulation rather than of one body. `g_constant` is the gravitational constant `G` used for the mutual attraction between every pair of bodies, `F = G m1 m2 / r^2`. **Its default is 1, not 0**, which catches people out: bodies rattling in a box also attract each other unless you say otherwise. Set it to 0 for a mechanism or a collision experiment; set it to a real value for an orbit.

The value expected here is `0`. A successful `set` prints nothing.

In [2]:
sim.do("set system.g_constant = 0")

### 6.2 Create the bodies

The next cell sends these 4 commands to the simulator. Here is exactly what each one expects:

---

`new sphere` creates one rigid body whose shape is **a uniform solid ball**, named `incomer`. The general form is

```
new <shape> [as <name>] { <field> = <value>, ... }
```

Its inertia tensor is not typed in — the simulator computes it from the shape: `I = diag(2mr^2/5, 2mr^2/5, 2mr^2/5)` — isotropic, so every axis through the centre is a principal axis with the same moment. The only shape whose inertia tensor is unchanged by any rotation, which makes it the simplest collidable body.

The fields given here are:

- `mass = 1` — the body's mass, in kilograms. Setting it also sets `inverse_mass` to its reciprocal — the two are kept consistent, and the simulator carries the inverse because that is what the equations of motion actually use.
- `radius = 0.5` — the radius, in metres.
- `position = [-3, 0, 0]` — the centre of mass, as `[x, y, z]` in metres.
- `velocity = [1, 0, 0]` — the centre-of-mass velocity, as `[vx, vy, vz]` in metres per second. Setting it sets the linear momentum to `m * v`; the momentum is what the state vector actually carries.

The command prints the name the simulator assigned, which is how you confirm the body exists.

---

`new sphere` creates one rigid body whose shape is **a uniform solid ball**, named `s1`. The general form is

```
new <shape> [as <name>] { <field> = <value>, ... }
```

Its inertia tensor is not typed in — the simulator computes it from the shape: `I = diag(2mr^2/5, 2mr^2/5, 2mr^2/5)` — isotropic, so every axis through the centre is a principal axis with the same moment. The only shape whose inertia tensor is unchanged by any rotation, which makes it the simplest collidable body.

The fields given here are:

- `mass = 1` — the body's mass, in kilograms. Setting it also sets `inverse_mass` to its reciprocal — the two are kept consistent, and the simulator carries the inverse because that is what the equations of motion actually use.
- `radius = 0.5` — the radius, in metres.
- `position = [0, 0, 0]` — the centre of mass, as `[x, y, z]` in metres.

The command prints the name the simulator assigned, which is how you confirm the body exists.

---

`new sphere` creates one rigid body whose shape is **a uniform solid ball**, named `s2`. The general form is

```
new <shape> [as <name>] { <field> = <value>, ... }
```

Its inertia tensor is not typed in — the simulator computes it from the shape: `I = diag(2mr^2/5, 2mr^2/5, 2mr^2/5)` — isotropic, so every axis through the centre is a principal axis with the same moment. The only shape whose inertia tensor is unchanged by any rotation, which makes it the simplest collidable body.

The fields given here are:

- `mass = 1` — the body's mass, in kilograms. Setting it also sets `inverse_mass` to its reciprocal — the two are kept consistent, and the simulator carries the inverse because that is what the equations of motion actually use.
- `radius = 0.5` — the radius, in metres.
- `position = [1, 0, 0]` — the centre of mass, as `[x, y, z]` in metres.

The command prints the name the simulator assigned, which is how you confirm the body exists.

---

`new sphere` creates one rigid body whose shape is **a uniform solid ball**, named `s3`. The general form is

```
new <shape> [as <name>] { <field> = <value>, ... }
```

Its inertia tensor is not typed in — the simulator computes it from the shape: `I = diag(2mr^2/5, 2mr^2/5, 2mr^2/5)` — isotropic, so every axis through the centre is a principal axis with the same moment. The only shape whose inertia tensor is unchanged by any rotation, which makes it the simplest collidable body.

The fields given here are:

- `mass = 1` — the body's mass, in kilograms. Setting it also sets `inverse_mass` to its reciprocal — the two are kept consistent, and the simulator carries the inverse because that is what the equations of motion actually use.
- `radius = 0.5` — the radius, in metres.
- `position = [2, 0, 0]` — the centre of mass, as `[x, y, z]` in metres.

The command prints the name the simulator assigned, which is how you confirm the body exists.

In [3]:
sim.do("new sphere as incomer { mass = 1, radius = 0.5, position = [-3, 0, 0], velocity = [1, 0, 0] }")
sim.do("new sphere as s1 { mass = 1, radius = 0.5, position = [0, 0, 0] }")
sim.do("new sphere as s2 { mass = 1, radius = 0.5, position = [1, 0, 0] }")
sim.do("new sphere as s3 { mass = 1, radius = 0.5, position = [2, 0, 0] }")

obj0 as incomer
obj1 as s1
obj2 as s2
obj3 as s3


### 6.3 Create the bodies (2)

The next cell sends this command to the simulator. Here is exactly what it expects:

---

`new sphere` creates one rigid body whose shape is **a uniform solid ball**, named `s4`. The general form is

```
new <shape> [as <name>] { <field> = <value>, ... }
```

Its inertia tensor is not typed in — the simulator computes it from the shape: `I = diag(2mr^2/5, 2mr^2/5, 2mr^2/5)` — isotropic, so every axis through the centre is a principal axis with the same moment. The only shape whose inertia tensor is unchanged by any rotation, which makes it the simplest collidable body.

The fields given here are:

- `mass = 1` — the body's mass, in kilograms. Setting it also sets `inverse_mass` to its reciprocal — the two are kept consistent, and the simulator carries the inverse because that is what the equations of motion actually use.
- `radius = 0.5` — the radius, in metres.
- `position = [3, 0, 0]` — the centre of mass, as `[x, y, z]` in metres.

The command prints the name the simulator assigned, which is how you confirm the body exists.

In [4]:
sim.do("new sphere as s4 { mass = 1, radius = 0.5, position = [3, 0, 0] }")

obj4 as s4


### 6.4 Set up the world (2)

The next cell sends this command to the simulator. Here is exactly what it expects:

---

`collide` arms contact detection. Impacts are then found as **roots** of the separation function by SUNDIALS' rootfinder, not by checking for overlap after the fact — so the time of impact is located to solver tolerance rather than to within a step, and fast bodies cannot tunnel through thin ones. The form is `COLLIDE [ON|OFF]`.

It prints the resulting state and how many pairs can collide.

In [5]:
sim.do("collide")

collisions ON (10 collidable pair(s); 0 impulse(s) so far)


### 6.5 Measure

The next cell sends these 2 commands to the simulator. Here is exactly what each one expects:

---

`momentum` prints the total linear momentum, summed over every body. With no external force it is exactly conserved, and it is conserved through collisions too, because every impulse is applied equal and opposite to the pair. It takes no arguments.

---

`energy` prints the total kinetic plus potential energy of the whole system. Take it before and after a run: for a closed system with no dissipation the two should agree to near machine precision, and how well they agree is the honest measure of the integration. It takes no arguments.

In [6]:
sim.do("momentum")
sim.do("energy")

[1, 0, 0]
0.5


### 6.6 Integrate

The next cell sends this command to the simulator. Here is exactly what it expects:

---

`run 4` advances the simulation by **4 seconds of duration** — not to absolute time 4. This trips people up: `run 1.7` followed by `run 2.89` leaves you at t = 4.59, not at 2.89. The form is `RUN <duration> [STEPS <n>]`.

`steps 1` asks for the result to be reported at 1 equally spaced points across that interval. It does not set the integrator's internal step size — SUNDIALS chooses that itself from the error tolerances, and takes as many internal steps between reports as accuracy demands.

This is the cell that actually integrates, so it is the slow one. It prints the time reached and how many internal solver steps were needed.

In [7]:
sim.do("run 4 steps 1")

t = 4 (18 solver steps, 1 snapshots, |dE/E| = 0.000e0, 4 collision(s) — CONTACTS lists them)


### 6.7 Measure (2)

The next cell sends these 4 commands to the simulator. Here is exactly what each one expects:

---

`get` evaluates an expression and prints it. The form is `GET <expression>`, where an expression may name a body's field (`obj0.position`, `bob.velocity.x`), index into it, or combine several with arithmetic and the built-in functions `norm`, `normalize`, `dot`, `cross`, `sqrt` and the rest.

Here it evaluates:

```
incomer.velocity.x
```

The corresponding Python call `sim.get(...)` returns the value as a Python object instead of printing it, which is what you want when the number is about to be used in a calculation.

---

`get` evaluates an expression and prints it. The form is `GET <expression>`, where an expression may name a body's field (`obj0.position`, `bob.velocity.x`), index into it, or combine several with arithmetic and the built-in functions `norm`, `normalize`, `dot`, `cross`, `sqrt` and the rest.

Here it evaluates:

```
s1.velocity.x
```

The corresponding Python call `sim.get(...)` returns the value as a Python object instead of printing it, which is what you want when the number is about to be used in a calculation.

---

`get` evaluates an expression and prints it. The form is `GET <expression>`, where an expression may name a body's field (`obj0.position`, `bob.velocity.x`), index into it, or combine several with arithmetic and the built-in functions `norm`, `normalize`, `dot`, `cross`, `sqrt` and the rest.

Here it evaluates:

```
s2.velocity.x
```

The corresponding Python call `sim.get(...)` returns the value as a Python object instead of printing it, which is what you want when the number is about to be used in a calculation.

---

`get` evaluates an expression and prints it. The form is `GET <expression>`, where an expression may name a body's field (`obj0.position`, `bob.velocity.x`), index into it, or combine several with arithmetic and the built-in functions `norm`, `normalize`, `dot`, `cross`, `sqrt` and the rest.

Here it evaluates:

```
s3.velocity.x
```

The corresponding Python call `sim.get(...)` returns the value as a Python object instead of printing it, which is what you want when the number is about to be used in a calculation.

In [8]:
sim.do("get incomer.velocity.x")
sim.do("get s1.velocity.x")
sim.do("get s2.velocity.x")
sim.do("get s3.velocity.x")

0
0
0
0


### 6.8 Measure (3)

The next cell sends these 4 commands to the simulator. Here is exactly what each one expects:

---

`get` evaluates an expression and prints it. The form is `GET <expression>`, where an expression may name a body's field (`obj0.position`, `bob.velocity.x`), index into it, or combine several with arithmetic and the built-in functions `norm`, `normalize`, `dot`, `cross`, `sqrt` and the rest.

Here it evaluates:

```
s4.velocity.x
```

The corresponding Python call `sim.get(...)` returns the value as a Python object instead of printing it, which is what you want when the number is about to be used in a calculation.

---

`momentum` prints the total linear momentum, summed over every body. With no external force it is exactly conserved, and it is conserved through collisions too, because every impulse is applied equal and opposite to the pair. It takes no arguments.

---

`energy` prints the total kinetic plus potential energy of the whole system. Take it before and after a run: for a closed system with no dissipation the two should agree to near machine precision, and how well they agree is the honest measure of the integration. It takes no arguments.

---

`contacts` lists every contact recorded so far, each with the exact time of impact found by the rootfinder, the contact normal, the point, and the impulse delivered. It takes no arguments.

In [9]:
sim.do("get s4.velocity.x")
sim.do("momentum")
sim.do("energy")
sim.do("contacts")

1
[1, 0, 0]
0.5
contact0: obj0 <-> obj1 at t = 2
  point  = [-0.5, 0, 0]
  normal = [1, 0, 0]  (from obj0 toward obj1)
  depth = 0, approach speed = 1, impulse = 1
contact1: obj1 <-> obj2 at t = 2
  point  = [0.5, 0, 0]
  normal = [1, 0, 0]  (from obj1 toward obj2)
  depth = 0, approach speed = 1, impulse = 1
contact2: obj2 <-> obj3 at t = 2
  point  = [1.5, 0, 0]
  normal = [1, 0, 0]  (from obj2 toward obj3)
  depth = 0, approach speed = 1, impulse = 1
contact3: obj3 <-> obj4 at t = 2
  point  = [2.5, 0, 0]
  normal = [1, 0, 0]  (from obj3 toward obj4)
  depth = 0, approach speed = 1, impulse = 1


### 6.9 Run the compiled Rust example this notebook pairs with

The next cell leaves the simulator aside and runs the **compiled example itself**, exactly as you would from a terminal:

```bash
cargo run --release -p physical_object --example newtons_cradle
```

`cargo run` builds the example if it is not already built (the first time, that can take a few minutes) and then executes it. The program sets up the same physics you just stepped through, integrates it with the same pure-Rust SUNDIALS translation, compares the results against its built-in closed-form expectations, and prints a final line beginning with `SUCCESS` or `FAILURE`, exiting nonzero on failure.

The Python here captures that output and re-prints it, then asserts that the verdict was `SUCCESS` — so if the compiled example and this notebook ever disagreed, running the notebook would fail loudly.

In [10]:
import subprocess, shutil
from pathlib import Path

# Run from the workspace root, wherever this notebook
# was started from. The workspace root is the ancestor that has both
# a Cargo.toml and the physical_object crate.
root = Path.cwd()
for cand in [root, *root.parents]:
    if (cand / 'Cargo.toml').is_file() and (cand / 'physical_object').is_dir():
        root = cand
        break

proc = subprocess.run(
    ['cargo', 'run', '--release', '-p', 'physical_object', '--example', 'newtons_cradle'],
    cwd=root, capture_output=True, text=True, timeout=1800,
)
print(proc.stdout)
if proc.returncode != 0:
    print(proc.stderr)
assert proc.returncode == 0, 'the compiled example reported FAILURE'
assert 'SUCCESS' in proc.stdout
print('the compiled example and this notebook agree')

Newton's cradle: 1 incomer at v = 1 into 4 touching spheres
  collisions resolved : 4
  final velocities    : [0.0, 0.0, 0.0, 0.0, 1.0]
  total momentum      : [1, 0, 0]
SUCCESS: impulse propagated through the chain, momentum conserved

the compiled example and this notebook agree


## 7. What those numbers mean

**What the cells above actually established.**

- `energy` was printed. Compare the value before the run with the value after it. For a closed system with no dissipation they should agree to near machine precision, and the size of the disagreement is the honest measure of the integration — not a number to be talked around.
- `momentum` was printed. With no external force the total linear momentum is exactly conserved, collisions included, because every impulse is applied equal and opposite.
- `contacts` was printed. Each entry carries the exact time of impact found by the rootfinder, the normal, the point and the impulse. The time of impact is accurate to solver tolerance, not to within one step.

**How to judge a number here.** A disagreement with theory is only a defect if it fails to shrink when you ask for more accuracy. Tighten `system.rtol` and `system.atol` and re-run: a discretisation error converges, and a bug does not. The one place this does not apply is a model with orientation-gripping joints, where `rtol` is floored at `1e-6` and tightening past it buys nothing.

**The full note from the example this notebook pairs with**, which sets out why it was built this way:

Newton's cradle: one incomer into a chain of four touching spheres.
Only the FAR sphere leaves, at the incomer's speed, and the four it
passed through stay exactly where they were.

That result is not a special rule about cradles. It is what momentum
and energy conservation force when the impacts are elastic and resolved
pairwise along the line of centres -- here by the collision solver's
Gauss-Seidel pass over the simultaneous contacts, with every impulse
applied equal and opposite so the total momentum is untouched.

Five unit spheres of radius 0.5. Four sit touching at x = 0, 1, 2, 3;
the incomer starts back at x = -3 moving at v = 1.


## 8. Close the simulator

The simulator is a separate operating-system process that this notebook
started. Closing it releases that process and its two pipes. If you skip
this cell the process is closed anyway when the Jupyter kernel shuts
down, but doing it explicitly means you can re-run the notebook from the
top without leaving a stray process behind each time.


In [11]:
sim.close()

simulator stopped


## 9. Name this notebook, and choose where to save it

Repeated in full in every notebook of this project.

The next cell asks you for two things:

1. **A name for this notebook.** It is typed into a normal input box that
   appears just under the cell. Press Enter to accept the suggestion shown
   in brackets.
2. **A folder to save it in.** A graphical "Save As" window opens on top of
   your other windows. Pick a folder, adjust the file name if you like, and
   press Save.

The graphical window is drawn by `tkinter`, which is part of Python's own
standard library, so nothing needs installing. If you are running this
notebook on a machine with no graphical display — over a plain SSH
connection, for instance — that window cannot open. The cell detects this,
says so, and falls back to asking you to type the folder path instead. It
will not fail or hang.

Saving writes a **copy** of this notebook, exactly as it stands now,
including every output you have produced. The original file is left
untouched.


In [ ]:
import json, os, shutil, sys
from pathlib import Path

# ---- 1. Ask the user to name the notebook -------------------------------
suggested = "rust_newtons_cradle.ipynb"
typed = input(f"Name for this notebook [{suggested}]: ").strip()
notebook_name = typed or suggested
if not notebook_name.endswith(".ipynb"):
    notebook_name += ".ipynb"
print(f"name chosen: {notebook_name}")

# ---- 2. Ask where to save it, with a pop-up dialog ----------------------
def ask_folder_graphically(initial_name):
    """Open a real Save-As window. Returns a path, or None if impossible."""
    try:
        import tkinter as tk
        from tkinter import filedialog
    except Exception as exc:
        print(f"(no tkinter available: {exc})")
        return None
    if sys.platform.startswith("linux") and not os.environ.get("DISPLAY"):
        print("(no graphical display detected: $DISPLAY is not set)")
        return None
    try:
        root = tk.Tk()
        root.withdraw()                 # hide the empty parent window
        root.attributes("-topmost", True)   # put the dialog in front
        chosen = filedialog.asksaveasfilename(
            title="Save this notebook as...",
            initialfile=initial_name,
            defaultextension=".ipynb",
            filetypes=[("Jupyter notebook", "*.ipynb"), ("All files", "*.*")],
        )
        root.destroy()
        return chosen or None
    except Exception as exc:
        print(f"(could not open the dialog: {exc})")
        return None

target = ask_folder_graphically(notebook_name)

if target is None:
    # Fallback for a machine with no display: ask in plain text instead.
    print("Falling back to a typed folder path.")
    default_dir = str(Path.cwd())
    folder = input(f"Folder to save into [{default_dir}]: ").strip() or default_dir
    target = str(Path(folder).expanduser() / notebook_name)

target = Path(target).expanduser()
target.parent.mkdir(parents=True, exist_ok=True)

# ---- 3. Copy this notebook to the chosen place --------------------------
source = Path("notebooks/rust_newtons_cradle.ipynb")
if not source.is_file():
    # The notebook may have been opened from somewhere else; look nearby.
    hits = list(Path.cwd().rglob(source.name))
    source = hits[0] if hits else None

if source is None:
    print(
        "Could not locate this notebook's own file on disk to copy it.\n"
        f"Use JupyterLab's  File -> Save Notebook As...  and save to:\n    {target}"
    )
else:
    shutil.copyfile(source, target)
    print(f"saved: {target}")